1. Load the prepared dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df_full = pd.read_excel('/content/drive/MyDrive/df_full.xlsx')

# Display the first 5 rows of the DataFrame to confirm it loaded correctly
display(df_full.head())

,parent_asin,title,main_category,features,description,details,average_rating,top_reviews
0,B09KH866S8,"Onamicit Key Finder,4 Pack Smart Bluetooth Tra...",Cell Phones & Accessories,['4 Key Finders：Our key finder is very conveni...,['This anti-lost device is a product based on ...,"[('Package Dimensions', '""4.17 x 3.39 x 0.79 i...",2.8,"[""This device sets the alarm if I close the ap..."
1,B09NYMK4Z2,"Celestron SCT To 2"" Adapter, Black",Camera & Photo,['Replace your 1.25” visual back with this SCT...,"['This sturdy, high-quality adapter provides a...","[('Product Dimensions', '""3 x 3 x 3 inches""'),...",4.6,['did exactly like advertised and was east to ...
2,B07PVYXPCL,"Sony A6400 Case, BolinUS Fullbody Ultra-Thin L...",Cell Phones & Accessories,['Designed Specifically for Sony Alpha A6400'\...,"['Protect your Sony Alpha A6400 from dust, scr...","[('Package Dimensions', '""4.4 x 3.8 x 1.6 inch...",4.2,['cover does fit ok a little lose in places su...
3,B07R3Z12VX,Coobes Compatible with Fitbit Inspire/Fitbit I...,Cell Phones & Accessories,['Perfect accessory case with screen protector...,['Coobes Compatible with Fitbit Inspire/Fitbit...,"[('Package Dimensions', '""5.08 x 3.03 x 0.51 i...",3.8,"['When I first installed it, it was great, the..."
4,B07SRXV9LG,"OMEN by HP 2019 15-Inch Gaming Laptop, Intel i...",Computers,['Hyper-realistic Graphics: NVIDIA GeForce RTX...,"[""You've got places to go, plays to make, and ...","[('Standing screen display size', '""15.6 Inche...",4.1,['Model: 15-dc1047nr<br /><br />Product: 7JG58...


In [ ]:
# Configuration
"""Shared configuration for the Electronics KB pipeline."""

# ---- Input / checkpoint paths ----
REVIEWS_CLEAN_PATH = "reviews_clean.parquet"        # output of step1
CHUNKS_PATH = "chunks.parquet"                      # output of step3
CORPUS_STATS_PATH = "corpus_stats.json"             # output of step5

# ---- Cleaning thresholds ----
LANG_KEEP = "en"

# ---- PII entities to mask ----
PII_ENTITIES = ["PHONE_NUMBER", "EMAIL_ADDRESS"]

# ---- Aspect taxonomy for semantic chunking ----
ASPECTS = [
    "price/value", "build quality", "specifications/performance",
    "battery life", "screen/display", "connectivity",
    "customer service", "ease of use", "sound quality", "design/appearance",
]

# ---- Indexing ----
ES_HOST = "http://localhost:9200"
ES_INDEX_NAME = "electronics_kb"
CHROMA_PATH = "./chroma_db"
CHROMA_COLLECTION = "electronics_kb"
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

# ---- Retrieval fusion ----
RRF_K = 60

TOP_K_DEFAULT = 10

2. Data Sanitization
* Deduplication (parent_asin)
* Language detection (English only)
* PII Masking (Names, Phone numbers)
* Microtext: Fix typos & slang

In [ ]:
!pip install fasttext presidio-analyzer presidio-anonymizer contractions spacy -q
!python -m spacy download en_core_web_sm -q
!wget -q https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("spaCy loaded OK")

spaCy loaded OK


In [ ]:
import re
import numpy as np
import pandas as pd
from config import REVIEWS_CLEAN_PATH, LANG_KEEP, PII_ENTITIES

# ------------------------------------------------------------
# If top_reviews was saved to xlsx as a stringified list
# (e.g. "['review one', 'review two']"), it needs parsing back
# into a real list before exploding.
# ------------------------------------------------------------
import ast

def parse_reviews_field(x):
    if isinstance(x, list):
        return x
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

df_full["top_reviews"] = df_full["top_reviews"].apply(parse_reviews_field)
reviews = df_full.explode("top_reviews").rename(columns={"top_reviews": "review_text"})
reviews = reviews.dropna(subset=["review_text"]).reset_index(drop=True)
reviews["review_text"] = reviews["review_text"].astype(str)
print(f"Exploded reviews: {len(reviews)}")

# ------------------------------------------------------------
# 1a. Deduplication (parent_asin + review text)
# ------------------------------------------------------------
before = len(reviews)
reviews = reviews.drop_duplicates(subset=["parent_asin", "review_text"]).reset_index(drop=True)
print(f"Dedup: {before} -> {len(reviews)}")

# ------------------------------------------------------------
# 1b. Strip video-tag placeholders
# ------------------------------------------------------------
VIDEO_TAG_PATTERN = re.compile(r"\[+\s*VIDEOID\s*:\s*[a-f0-9]+\s*\]+", flags=re.IGNORECASE)
reviews["review_text"] = reviews["review_text"].apply(lambda t: VIDEO_TAG_PATTERN.sub("", t).strip())
reviews = reviews[reviews["review_text"] != ""].reset_index(drop=True)
print(f"After video-tag strip: {len(reviews)}")


Exploded reviews: 1803756
Dedup: 1803756 -> 1783584
After video-tag strip: 1783584


In [ ]:
# ------------------------------------------------------------
# 1c. Language detection (English only) — fastText
# ------------------------------------------------------------
import fasttext

lang_model = fasttext.load_model("lid.176.bin")

# monkeypatch for numpy 2.0 incompatibility in fasttext's predict()
from fasttext.FastText import _FastText
def _patched_predict(self, text, k=1, threshold=0.0, on_unicode_error="strict"):
    def check(entry):
        if entry.find("\n") != -1:
            raise ValueError("predict processes one line at a time (remove '\\n')")
        return entry + "\n"
    text = check(text)
    predictions = self.f.predict(text, k, threshold, on_unicode_error)
    if len(predictions):
        probs, labels = zip(*predictions)
    else:
        probs, labels = ([], ())
    return labels, np.asarray(probs)
_FastText.predict = _patched_predict

def detect_lang(text: str) -> str:
    clean = text.replace("\n", " ").strip()
    if not clean:
        return "unknown"
    label, _ = lang_model.predict(clean, k=1)
    return label[0].replace("__label__", "") if label else "unknown"

reviews["lang"] = reviews["review_text"].apply(detect_lang)
print(reviews["lang"].value_counts().head(10))

before = len(reviews)
reviews = reviews[reviews["lang"] == LANG_KEEP].drop(columns=["lang"]).reset_index(drop=True)
print(f"Language filter ({LANG_KEEP} only): {before} -> {len(reviews)}")

lang
en    1770864
es      11888
ja        490
de         82
fr         66
pt         62
ru         33
it         25
zh         20
ca         10
Name: count, dtype: int64
Language filter (en only): 1783584 -> 1770864


In [ ]:
# ------------------------------------------------------------
# 1d. PII masking (names, phone numbers, emails) — fast path
#     Regex for emails/phones/urls (cheap, catches most liability),
#     spaCy NER batched via nlp.pipe() for PERSON names (fast enough
#     at corpus scale, unlike per-string Presidio calls).
# ------------------------------------------------------------
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE_RE = re.compile(r"(\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b")
URL_RE = re.compile(r"https?://\S+|www\.\S+")

def mask_contact_info(text: str) -> str:
    text = EMAIL_RE.sub("[EMAIL]", text)
    text = PHONE_RE.sub("[PHONE]", text)
    text = URL_RE.sub("[URL]", text)
    return text

reviews["review_text"] = reviews["review_text"].apply(mask_contact_info)

import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "tagger", "lemmatizer"])

def mask_names_batch(texts, batch_size=256):
    out = []
    for text, doc in zip(texts, nlp.pipe(texts, batch_size=batch_size)):
        redacted = text
        for ent in reversed(doc.ents):
            if ent.label_ == "PERSON":
                redacted = redacted[:ent.start_char] + "[PERSON]" + redacted[ent.end_char:]
        out.append(redacted)
    return out

reviews["review_text"] = mask_names_batch(reviews["review_text"].tolist())
print("PII masking complete (emails, phones, URLs, person names)")

KeyboardInterrupt: 

It took more than 2 hours for the PII masking step to complete, so the following is another alternative to keep personal info safe:

In [ ]:
# Check if there is sensitive info first
sample = reviews["review_text"].sample(2000, random_state=42)
has_email = sample.str.contains(EMAIL_RE)
has_phone = sample.str.contains(PHONE_RE, regex=True)
print(f"Emails: {has_email.sum()}/2000, Phones: {has_phone.sum()}/2000")

Emails: 0/2000, Phones: 0/2000


/tmp/ipykernel_12970/2711354141.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_phone = sample.str.contains(PHONE_RE, regex=True)


In [ ]:
# mask any sensitive info with pure text, such as "[Email]", "[Phone]"
reviews["review_text"] = reviews["review_text"].apply(mask_contact_info)

In [ ]:
# ------------------------------------------------------------
# 1e. Microtext normalization: contractions + curated slang map
#     (deliberately NOT a generic spellchecker — that mangles
#     model numbers / brand names like "4K", "iPhone13")
# ------------------------------------------------------------
import contractions

SLANG_MAP = {
    r"\bw/\b": "with", r"\bw/o\b": "without", r"\bu\b": "you",
    r"\bur\b": "your", r"\bpls\b": "please", r"\bthx\b": "thanks",
    r"\btho\b": "though", r"\bbc\b": "because", r"\bidk\b": "I don't know",
    r"\bimo\b": "in my opinion", r"\bdef\b": "definitely",
}
SLANG_PATTERNS = [(re.compile(p, re.IGNORECASE), r) for p, r in SLANG_MAP.items()]

def safe_fix_contractions(text: str) -> str:
    try:
        return contractions.fix(text)
    except (IndexError, Exception):
        return text  # leave unchanged if the library chokes on it

def normalize_text(text: str) -> str:
    text = safe_fix_contractions(text)
    for pattern, repl in SLANG_PATTERNS:
        text = pattern.sub(repl, text)
    return re.sub(r"\s+", " ", text).strip()

reviews["review_text"] = reviews["review_text"].apply(normalize_text)
print("Normalization complete")

Normalization complete


In [ ]:
# checking the result
sample_idx = reviews.sample(5, random_state=1).index
for i in sample_idx:
    print("AFTER: ", reviews.loc[i, "review_text"])
    print("---")

AFTER:  My child love it!!!it was a. Great gif t the look on her face was priceless I still feel harm inside from that look
---
AFTER:  If you are going to spend a couple of hundred dollars on a beautiful camera, you might as well spend a few more dollars on a strong beautiful leather case to protect your camera. I have received a lot of compliments regarding this case. I would recommend others to buy this as well!
---
AFTER:  I purchased 2 of these in June it is now October and only one still works. Garbage. Pay for the real thing do not waste your time with these. I am sure it will not be long for the other one to crap out
---
AFTER:  I am using it with a Samsung slate and a laptop. It has been working well, at least for the first week. I call it Recommended.
---
AFTER:  I am not sure about how these will hold up long term But these are only 3 dollars a piece and they do not look or feel like jump. Overall I like these, good colors and and work well.
---


In [ ]:
# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------
reviews = reviews.drop(columns=["word_count"], errors="ignore")
reviews.to_parquet(REVIEWS_CLEAN_PATH)
print(f"Saved {len(reviews)} cleaned reviews -> {REVIEWS_CLEAN_PATH}")

Saved 1770864 cleaned reviews -> reviews_clean.parquet


3. Semantic Chunking
* Split by Aspect (Price, Quality, Specs)
* Context injection (ASIN + Title per chunk)
* Metadata extraction (Rating, Category)
* Cross-linking reviews to features

In [ ]:
import ast
import json
import re
import time
import numpy as np
import pandas as pd
from config import REVIEWS_CLEAN_PATH, CHUNKS_PATH

In [ ]:
# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
df = pd.read_excel("/content/df_full.xlsx")
reviews = pd.read_parquet(REVIEWS_CLEAN_PATH)

nlp = spacy.load("en_core_web_sm")  # sentence splitting only
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)  # device=0 for GPU

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def split_sentences(text: str) -> list:
    return [s.text.strip() for s in nlp(text).sents if len(s.text.strip()) > 3]

def assign_aspect(sentence: str) -> str:
    result = zero_shot(sentence, candidate_labels=ASPECTS, multi_label=False)
    return result["labels"][0]

def flatten_details(details) -> str:
    if details is None:
        return ""
    if isinstance(details, np.ndarray):
        details = details.tolist()
    if isinstance(details, list):
        if len(details) == 0:
            return ""
        # list of (key, value) pairs, as seen in this dataset's `details` field
        return "; ".join(f"{k}: {v}" for k, v in details)
    if isinstance(details, dict):
        return "; ".join(f"{k}: {v}" for k, v in details.items())
    return str(details)

def flatten_list_field(x) -> str:
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, list):
        return " ".join(str(i) for i in x)
    return str(x) if x else ""

In [ ]:
# ------------------------------------------------------------
# 3a. Spec chunks — one per product, context-injected
# ------------------------------------------------------------
spec_chunks = []
for _, row in df.iterrows():
    parts = []
    features_text = flatten_list_field(row.get("features"))
    if features_text:
        parts.append(f"Features: {features_text}")
    details_text = flatten_details(row.get("details"))
    if details_text:
        parts.append(f"Specifications: {details_text}")
    desc_text = flatten_list_field(row.get("description"))
    if desc_text:
        parts.append(f"Description: {desc_text}")

    if not parts:
        continue

    # context injection: ASIN + title prefixed to every chunk
    title = row.get("title", "")
    context_prefix = f"[Product: {title} | ASIN: {row['parent_asin']}] "
    chunk_text = context_prefix + " | ".join(parts)

    spec_chunks.append({
        "chunk_id": f"{row['parent_asin']}_spec",
        "parent_asin": row["parent_asin"],
        "chunk_type": "spec",
        "aspect": "specifications",
        "text": chunk_text,
        "rating": row.get("average_rating"),
        "category": row.get("main_category"),
    })

spec_chunks_df = pd.DataFrame(spec_chunks)
print(f"Spec chunks: {len(spec_chunks_df)}")


Spec chunks: 198592


In [ ]:
# test for zero-shot run time control
import time

test_reviews = reviews["review_text"].head(50).tolist()

start = time.time()
test_sentences = []
for text in test_reviews:
    test_sentences.extend(split_sentences(text))

print(f"50 reviews -> {len(test_sentences)} sentences")

start = time.time()
for sent in test_sentences[:50]:  # time just 50 sentences through zero-shot
    assign_aspect(sent)
elapsed = time.time() - start
rate = 50 / elapsed
print(f"{rate:.2f} sentences/sec")

# estimate total sentence count and total time
avg_sentences_per_review = len(test_sentences) / len(test_reviews)
est_total_sentences = avg_sentences_per_review * len(reviews)
est_total_minutes = est_total_sentences / rate / 60
print(f"Estimated total sentences: {est_total_sentences:.0f}")
print(f"Estimated total time: {est_total_minutes:.1f} minutes ({est_total_minutes/60:.1f} hours)")

50 reviews -> 297 sentences


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


4.33 sentences/sec
Estimated total sentences: 10518932
Estimated total time: 40519.1 minutes (675.3 hours)


Because it took tremendous time to run zero-shot, we switched to keyword-based aspect assignment

In [ ]:
# ------------------------------------------------------------
# Aspect taxonomy — keyword-based (fast path)
# ------------------------------------------------------------
ASPECT_KEYWORDS = {
    "price/value": ["price", "cost", "value", "expensive", "cheap", "worth", "money", "overpriced", "affordable"],
    "battery life": ["battery", "charge", "charging", "power", "drain", "mah"],
    "build quality": ["quality", "sturdy", "durable", "broke", "broken", "flimsy", "solid", "cheaply made", "well made"],
    "screen/display": ["screen", "display", "resolution", "brightness", "pixel", "touchscreen"],
    "connectivity": ["bluetooth", "wifi", "wi-fi", "connect", "connection", "pairing", "usb", "port", "signal"],
    "sound quality": ["sound", "audio", "bass", "volume", "speaker", "mic", "microphone", "noise cancel"],
    "customer service": ["service", "support", "return", "refund", "warranty", "replace", "customer"],
    "ease of use": ["easy", "simple", "intuitive", "setup", "install", "user friendly", "instructions"],
    "design/appearance": ["design", "look", "color", "style", "appearance", "sleek", "bulky", "compact"],
    "specifications/performance": ["performance", "speed", "fast", "slow", "spec", "processor", "storage", "memory", "lag"],
}

def assign_aspect_keyword(sentence: str) -> str:
    sentence_lower = sentence.lower()
    best_aspect, best_count = "general", 0
    for aspect, keywords in ASPECT_KEYWORDS.items():
        count = sum(1 for kw in keywords if kw in sentence_lower)
        if count > best_count:
            best_aspect, best_count = aspect, count
    return best_aspect

def split_sentences(text: str) -> list:
    return [s.text.strip() for s in nlp(text).sents if len(s.text.strip()) > 3]

In [ ]:
# ------------------------------------------------------------
# 3b. Review chunks — aspect-split per review, context-injected
# ------------------------------------------------------------
title_lookup = df.set_index("parent_asin")["title"].to_dict()
rating_lookup = df.set_index("parent_asin")["average_rating"].to_dict()
category_lookup = df.set_index("parent_asin")["main_category"].to_dict()

start = time.time()
review_chunks = []
chunk_counter = 0

for row in reviews.itertuples(index=False):
    asin = row.parent_asin
    sentences = split_sentences(row.review_text)
    if not sentences:
        continue

    aspect_groups = {}
    for sent in sentences:
        aspect = assign_aspect_keyword(sent)
        aspect_groups.setdefault(aspect, []).append(sent)

    title = title_lookup.get(asin, "")
    context_prefix = f"[Product: {title} | ASIN: {asin}] "
    rating = rating_lookup.get(asin)
    category = category_lookup.get(asin)

    for aspect, sents in aspect_groups.items():
        review_chunks.append({
            "chunk_id": f"{asin}_review_{chunk_counter}",
            "parent_asin": asin,
            "chunk_type": "review",
            "aspect": aspect,
            "text": context_prefix + " ".join(sents),
            "rating": rating,
            "category": category,
        })
        chunk_counter += 1

    # progress heartbeat every 100k reviews, so a long run isn't silent
    if chunk_counter and chunk_counter % 100_000 < len(sentences):
        elapsed_so_far = time.time() - start
        print(f"  ...{chunk_counter} chunks so far, {elapsed_so_far:.0f}s elapsed")

elapsed = time.time() - start
print(f"Review chunking took {elapsed:.1f}s for {len(reviews)} reviews "
      f"({len(reviews)/elapsed:.0f} reviews/sec)")

review_chunks_df = pd.DataFrame(review_chunks)
print(f"Review chunks: {len(review_chunks_df)}")


  ...1 chunks so far, 0s elapsed
  ...5 chunks so far, 0s elapsed
  ...100001 chunks so far, 732s elapsed
  ...200000 chunks so far, 1466s elapsed
  ...200002 chunks so far, 1466s elapsed
  ...300003 chunks so far, 2197s elapsed
  ...400001 chunks so far, 2921s elapsed
  ...400010 chunks so far, 2921s elapsed
  ...500001 chunks so far, 3652s elapsed
  ...500013 chunks so far, 3652s elapsed
  ...500018 chunks so far, 3652s elapsed
  ...600001 chunks so far, 4390s elapsed
  ...600004 chunks so far, 4390s elapsed
  ...700000 chunks so far, 5154s elapsed
  ...700001 chunks so far, 5154s elapsed
  ...700005 chunks so far, 5154s elapsed
  ...800003 chunks so far, 6005s elapsed
  ...800008 chunks so far, 6005s elapsed
  ...900000 chunks so far, 6860s elapsed
  ...900003 chunks so far, 6860s elapsed
  ...1000003 chunks so far, 7709s elapsed
  ...1100000 chunks so far, 8564s elapsed
  ...1100003 chunks so far, 8564s elapsed
  ...1100013 chunks so far, 8564s elapsed
  ...1200001 chunks so far, 9

In [ ]:
print(len(review_chunks))

4252157


In [ ]:
# ------------------------------------------------------------
# 3c. Combine + cross-link (shared parent_asin across chunk_types)
# ------------------------------------------------------------
all_chunks = pd.concat([spec_chunks_df, review_chunks_df], ignore_index=True)
all_chunks.to_parquet(CHUNKS_PATH)
print(f"Saved {len(all_chunks)} total chunks -> {CHUNKS_PATH}")
print(all_chunks["chunk_type"].value_counts())
print(all_chunks["aspect"].value_counts())

Saved 4450749 total chunks -> chunks.parquet
chunk_type
review    4252157
spec       198592
Name: count, dtype: int64
aspect
general                       1534726
price/value                    460441
connectivity                   313514
build quality                  312062
design/appearance              302166
battery life                   285714
ease of use                    276291
specifications/performance     227266
customer service               217752
specifications                 198592
screen/display                 185481
sound quality                  136744
Name: count, dtype: int64


In [ ]:
# save the result
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("chunks.parquet", "/content/drive/MyDrive/chunks.parquet")

Mounted at /content/drive


'/content/drive/MyDrive/chunks.parquet'

4. Indexing
* Elasticsearch: BM25 for keywords
* ChromaDB/Qdrant: Dense embeddings
* Reciprocal Rank Fusion (RRF) setup
* Metadata HNSW Indexing

In [ ]:
!pip install elasticsearch
!pip install chromadb
!pip install sentence_transformers
!pip install rank_bm25

In [ ]:
import pickle
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
import chromadb
from sentence_transformers import SentenceTransformer
from config import CHUNKS_PATH, CHROMA_PATH, CHROMA_COLLECTION, EMBED_MODEL_NAME, RRF_K, TOP_K_DEFAULT

BM25_INDEX_PATH = "bm25_index.pkl"

chunks = pd.read_parquet("/content/chunks.parquet")
chunks["chunk_id"] = chunks["chunk_id"].astype(str)
chunk_id_to_pos = {cid: i for i, cid in enumerate(chunks["chunk_id"])}  # for fast row lookup

In [ ]:
# --------------------------------------------------------------------------------
# 4a. BM25 — in-process keyword index (replaces Elasticsearch due to access issue)
# --------------------------------------------------------------------------------
print("Tokenizing corpus for BM25...")
tokenized_corpus = [str(doc).lower().split() for doc in chunks["text"]]
bm25 = BM25Okapi(tokenized_corpus)

# persist so you don't have to retokenize/rebuild on every session
with open(BM25_INDEX_PATH, "wb") as f:
    pickle.dump({"bm25": bm25, "chunk_ids": chunks["chunk_id"].tolist()}, f)
print(f"BM25 index built over {len(chunks)} chunks -> saved to {BM25_INDEX_PATH}")

def bm25_search(query: str, k: int = 50) -> list:
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_positions = np.argsort(scores)[::-1][:k]
    return chunks.iloc[top_positions]["chunk_id"].tolist()

Tokenizing corpus for BM25...
BM25 index built over 4450749 chunks -> saved to bm25_index.pkl


In [ ]:
# save the result
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("bm25_index.pkl", "/content/drive/MyDrive/bm25_index.pkl")

Mounted at /content/drive


'/content/drive/MyDrive/bm25_index.pkl'

In [ ]:
# For word embedding, force the model onto GPU explicitly to save run time
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")

True NVIDIA L4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
import time
test_batch = chunks.iloc[:2000]
start = time.time()
_ = embed_model.encode(test_batch["text"].tolist(), batch_size=128, show_progress_bar=False)
rate = 2000 / (time.time() - start)
print(f"{rate:.0f} chunks/sec -> estimated total: {len(chunks)/rate/60:.1f} minutes")

72 chunks/sec -> estimated total: 1037.4 minutes


To reduce time, we pre-process chunks as follows:

In [ ]:
# chunk stats
lengths = chunks["text"].str.len()
print(lengths.describe())
print(f"Chunks over 1000 chars: {(lengths > 1000).sum()}")

count    4.450749e+06
mean     4.205647e+02
std      5.971880e+02
min      4.400000e+01
25%      2.050000e+02
50%      2.730000e+02
75%      3.920000e+02
max      3.702600e+04
Name: text, dtype: float64
Chunks over 1000 chars: 281560


It is observed that chunk distribution is highly skewed

In [ ]:
# fix chunk length limit
MAX_CHARS = 2000  # roughly ~400-500 tokens for typical English text, safely under the 512 limit

chunks["text"] = chunks["text"].apply(lambda t: t[:MAX_CHARS] if isinstance(t, str) else t)

In [ ]:
# sort by length before batching
chunks_sorted = chunks.copy()
chunks_sorted["text_len"] = chunks_sorted["text"].str.len()
chunks_sorted = chunks_sorted.sort_values("text_len").reset_index(drop=True)

In [ ]:
# re-test running time
import time
test_batch = chunks_sorted.iloc[2_000_000:2_002_000]  # grab a slice from the sorted middle, not just the start
start = time.time()
_ = embed_model.encode(test_batch["text"].tolist(), batch_size=256, show_progress_bar=False)
rate = 2000 / (time.time() - start)
print(f"{rate:.0f} chunks/sec -> estimated total: {len(chunks)/rate/60:.1f} minutes")

312 chunks/sec -> estimated total: 237.4 minutes


Now as the time has been largely reduced, we can do word embedding now.

In [ ]:
# ------------------------------------------------------------
# 4b. ChromaDB — dense vector index (HNSW under the hood, unchanged)
# ------------------------------------------------------------
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
print(f"Embedding model device: {embed_model.device}")  # should print cuda:0 — confirm before proceeding

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

BATCH = 2048       # outer loop batch (Chroma writes)
ENCODE_BATCH = 256  # inner encode() batch — matches the tested ~312 chunks/sec configuration

for start in range(0, len(chunks), BATCH):
    batch = chunks.iloc[start:start + BATCH]
    embeddings = embed_model.encode(
        batch["text"].tolist(),
        batch_size=ENCODE_BATCH,
        show_progress_bar=False,
    ).tolist()
    collection.add(
        ids=batch["chunk_id"].tolist(),
        embeddings=embeddings,
        documents=batch["text"].tolist(),
        metadatas=[
            {
                "parent_asin": r.parent_asin,
                "chunk_type": r.chunk_type,
                "aspect": r.aspect,
                "category": str(r.category),
                "rating": float(r.rating) if pd.notna(r.rating) else 0.0,
            }
            for r in batch.itertuples()
        ],
    )
    if start % (BATCH * 5) == 0:
        print(f"Embedded {start + len(batch)} / {len(chunks)}")

print("Indexed into ChromaDB")

def dense_search(query: str, k: int = 50) -> list:
    query_emb = embed_model.encode([query]).tolist()
    result = collection.query(query_embeddings=query_emb, n_results=k)
    return result["ids"][0]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model device: cuda:0
Embedded 2048 / 4450749
Embedded 12288 / 4450749
Embedded 22528 / 4450749
Embedded 32768 / 4450749
Embedded 43008 / 4450749
Embedded 53248 / 4450749
Embedded 63488 / 4450749
Embedded 73728 / 4450749
Embedded 83968 / 4450749
Embedded 94208 / 4450749
Embedded 104448 / 4450749
Embedded 114688 / 4450749
Embedded 124928 / 4450749
Embedded 135168 / 4450749
Embedded 145408 / 4450749
Embedded 155648 / 4450749
Embedded 165888 / 4450749
Embedded 176128 / 4450749
Embedded 186368 / 4450749
Embedded 196608 / 4450749
Embedded 206848 / 4450749
Embedded 217088 / 4450749
Embedded 227328 / 4450749
Embedded 237568 / 4450749
Embedded 247808 / 4450749
Embedded 258048 / 4450749
Embedded 268288 / 4450749
Embedded 278528 / 4450749
Embedded 288768 / 4450749
Embedded 299008 / 4450749
Embedded 309248 / 4450749
Embedded 319488 / 4450749
Embedded 329728 / 4450749
Embedded 339968 / 4450749
Embedded 350208 / 4450749
Embedded 360448 / 4450749
Embedded 370688 / 4450749
Embedded 380928 / 

In [ ]:
import os
import zipfile

with zipfile.ZipFile("electronics_kb_bundle.zip", "w") as zf:
    zf.write("chunks.parquet")
    zf.write("/content/drive/MyDrive/bm25_index.pkl")
    zf.write("config.py")
    for root, _, files in os.walk("chroma_db"):
        for f in files:
            path = os.path.join(root, f)
            zf.write(path, arcname=path)

print("Bundle created: electronics_kb_bundle.zip")

Bundle created: electronics_kb_bundle.zip


In [ ]:
# save the result
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("electronics_kb_bundle.zip", "/content/drive/MyDrive/electronics_kb_bundle.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/electronics_kb_bundle.zip'

checking

In [ ]:
import os
print(os.getcwd())
print(os.path.exists("chroma_db"))
!ls -la

/content
False
total 16
drwxr-xr-x 1 root root 4096 Sep  4 13:32 .
drwxr-xr-x 1 root root 4096 Sep 28 20:55 ..
drwxr-xr-x 4 root root 4096 Sep  4 13:32 .config
drwxr-xr-x 1 root root 4096 Sep  4 13:32 sample_data


In [ ]:
!find / -type d -name "chroma_db" 2>/dev/null

In [ ]:
"""
STEP 4b — RESUMABLE, SHARDED EMBEDDING (Drive-backed)

Colab runtime reset wiped /content, including the
Chroma index that took hours to build. This version writes embeddings in
shards to Google Drive (.npy + ids), so a reset only costs the shard that
was in progress. Re-running the script skips every shard already on Drive.

Flow:
  1. Mount Drive, load chunks.parquet (from Drive)
  2. Apply the same length-cap + sort used before (order must be identical
     between runs, so shard boundaries stay stable)
  3. Embed shard by shard -> save to Drive
  4. Build Chroma (local disk) from the saved shards — no GPU needed

Run in Colab with a GPU runtime.
"""

import os
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

# ---- paths ----
DRIVE_DIR = "/content/drive/MyDrive/electronics_kb"
CHUNKS_PATH = f"{DRIVE_DIR}/chunks.parquet"
SHARD_DIR = f"{DRIVE_DIR}/embedding_shards"
os.makedirs(SHARD_DIR, exist_ok=True)

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
MAX_CHARS = 2000
SHARD_SIZE = 100_000     # chunks per shard: ~5 min of work at ~312 chunks/sec
ENCODE_BATCH = 256

Mounted at /content/drive


In [ ]:
# ------------------------------------------------------------
# Load + deterministic ordering (must match on every run)
# ------------------------------------------------------------
chunks = pd.read_parquet(CHUNKS_PATH)
chunks["chunk_id"] = chunks["chunk_id"].astype(str)
chunks["text"] = chunks["text"].apply(lambda t: t[:MAX_CHARS] if isinstance(t, str) else t)
chunks["text_len"] = chunks["text"].str.len()
# stable sort + chunk_id tiebreak so ordering is identical across sessions
chunks = chunks.sort_values(["text_len", "chunk_id"], kind="stable").reset_index(drop=True)
chunks = chunks.drop(columns=["text_len"])

n_shards = (len(chunks) + SHARD_SIZE - 1) // SHARD_SIZE
print(f"{len(chunks)} chunks -> {n_shards} shards of up to {SHARD_SIZE}")

# ------------------------------------------------------------
# Embed shard by shard, skipping any shard already saved
# ------------------------------------------------------------
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
print(f"Embedding model device: {embed_model.device}")  # must say cuda:0

for shard_idx in range(n_shards):
    emb_path = f"{SHARD_DIR}/shard_{shard_idx:04d}_emb.npy"
    ids_path = f"{SHARD_DIR}/shard_{shard_idx:04d}_ids.npy"
    if os.path.exists(emb_path) and os.path.exists(ids_path):
        continue  # already done in an earlier session

    shard = chunks.iloc[shard_idx * SHARD_SIZE:(shard_idx + 1) * SHARD_SIZE]
    embeddings = embed_model.encode(
        shard["text"].tolist(),
        batch_size=ENCODE_BATCH,
        show_progress_bar=False,
        normalize_embeddings=True,
    ).astype(np.float16)  # halves Drive storage; negligible effect on retrieval

    # write ids first, embeddings last: a shard only counts as done when both exist
    np.save(ids_path, shard["chunk_id"].to_numpy(dtype=str))
    np.save(emb_path, embeddings)
    print(f"Saved shard {shard_idx + 1}/{n_shards}")

print("All shards embedded.")

4450749 chunks -> 45 shards of up to 100000


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model device: cuda:0
Saved shard 1/45
Saved shard 2/45
Saved shard 3/45
Saved shard 4/45
Saved shard 5/45
Saved shard 6/45
Saved shard 7/45
Saved shard 8/45
Saved shard 9/45
Saved shard 10/45
Saved shard 11/45
Saved shard 12/45
Saved shard 13/45
Saved shard 14/45
Saved shard 15/45
Saved shard 16/45
Saved shard 17/45
Saved shard 18/45
Saved shard 19/45
Saved shard 20/45
Saved shard 21/45
Saved shard 22/45
Saved shard 23/45
Saved shard 24/45
Saved shard 25/45
Saved shard 26/45
Saved shard 27/45
Saved shard 28/45
Saved shard 29/45
Saved shard 30/45
Saved shard 31/45
Saved shard 32/45
Saved shard 33/45
Saved shard 34/45
Saved shard 35/45
Saved shard 36/45
Saved shard 37/45
Saved shard 38/45
Saved shard 39/45
Saved shard 40/45
Saved shard 41/45
Saved shard 42/45
Saved shard 43/45
Saved shard 44/45
Saved shard 45/45
All shards embedded.


In [ ]:
!ls -la /content/drive/MyDrive/electronics_kb/embedding_shards

total 7110815
drwx------ 2 root root      4096 Sep 28 02:05 .
drwx------ 3 root root      4096 Sep 28 02:02 ..
-rw------- 1 root root 153600128 Sep 28 02:08 shard_0000_emb.npy
-rw------- 1 root root  10000128 Sep 28 02:08 shard_0000_ids.npy
-rw------- 1 root root 153600128 Sep 28 02:10 shard_0001_emb.npy
-rw------- 1 root root  10000128 Sep 28 02:10 shard_0001_ids.npy
-rw------- 1 root root 153600128 Sep 28 02:13 shard_0002_emb.npy
-rw------- 1 root root  10000128 Sep 28 02:13 shard_0002_ids.npy
-rw------- 1 root root 153600128 Sep 28 02:16 shard_0003_emb.npy
-rw------- 1 root root  10000128 Sep 28 02:16 shard_0003_ids.npy
-rw------- 1 root root 153600128 Sep 28 02:19 shard_0004_emb.npy
-rw------- 1 root root  10000128 Sep 28 02:19 shard_0004_ids.npy
-rw------- 1 root root 153600128 Sep 28 02:22 shard_0005_emb.npy
-rw------- 1 root root  10000128 Sep 28 02:22 shard_0005_ids.npy
-rw------- 1 root root 153600128 Sep 28 02:25 shard_0006_emb.npy
-rw------- 1 root root  10000128 Sep 28 02:2

In [ ]:
!pip install chromadb rank_bm25 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/electronics_kb"
CHUNKS_PATH = f"{DRIVE_DIR}/chunks.parquet"
SHARD_DIR = f"{DRIVE_DIR}/embedding_shards"

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
MAX_CHARS = 2000
SHARD_SIZE = 100_000
ENCODE_BATCH = 256

chunks = pd.read_parquet(CHUNKS_PATH)
chunks["chunk_id"] = chunks["chunk_id"].astype(str)
chunks["text"] = chunks["text"].apply(lambda t: t[:MAX_CHARS] if isinstance(t, str) else t)
chunks["text_len"] = chunks["text"].str.len()
chunks = chunks.sort_values(["text_len", "chunk_id"], kind="stable").reset_index(drop=True)
chunks = chunks.drop(columns=["text_len"])

n_shards = (len(chunks) + SHARD_SIZE - 1) // SHARD_SIZE
print(f"{len(chunks)} chunks -> {n_shards} shards")

Mounted at /content/drive
4450749 chunks -> 45 shards


In [ ]:
# ------------------------------------------------------------
# Build Chroma locally from the saved shards
# ------------------------------------------------------------
import chromadb

CHROMA_PATH = "./chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    "electronics_kb", metadata={"hnsw:space": "cosine"}
)

already = collection.count()
print(f"Chroma currently holds {already} chunks")

meta_lookup = chunks.set_index("chunk_id")
ADD_BATCH = 5000  # Chroma limits the max batch size per add() call

for shard_idx in range(n_shards):
    start = shard_idx * SHARD_SIZE
    end = min(start + SHARD_SIZE, len(chunks))
    if end <= already:
        continue  # this shard is already in Chroma

    ids = np.load(f"{SHARD_DIR}/shard_{shard_idx:04d}_ids.npy")
    emb = np.load(f"{SHARD_DIR}/shard_{shard_idx:04d}_emb.npy").astype(np.float32)
    rows = meta_lookup.loc[ids]

    for i in range(0, len(ids), ADD_BATCH):
        sl = slice(i, i + ADD_BATCH)
        sub = rows.iloc[sl]
        collection.add(
            ids=ids[sl].tolist(),
            embeddings=emb[sl].tolist(),
            documents=sub["text"].tolist(),
            metadatas=[
                {
                    "parent_asin": r.parent_asin,
                    "chunk_type": r.chunk_type,
                    "aspect": r.aspect,
                    "category": str(r.category),
                    "rating": float(r.rating) if pd.notna(r.rating) else 0.0,
                }
                for r in sub.itertuples()
            ],
        )
    print(f"Chroma: loaded shard {shard_idx + 1}/{n_shards}")

print(f"Done. Chroma holds {collection.count()} chunks.")

Chroma currently holds 0 chunks
Chroma: loaded shard 1/45
Chroma: loaded shard 2/45
Chroma: loaded shard 3/45
Chroma: loaded shard 4/45
Chroma: loaded shard 5/45
Chroma: loaded shard 6/45
Chroma: loaded shard 7/45
Chroma: loaded shard 8/45
Chroma: loaded shard 9/45
Chroma: loaded shard 10/45
Chroma: loaded shard 11/45
Chroma: loaded shard 12/45
Chroma: loaded shard 13/45
Chroma: loaded shard 14/45
Chroma: loaded shard 15/45
Chroma: loaded shard 16/45
Chroma: loaded shard 17/45
Chroma: loaded shard 18/45
Chroma: loaded shard 19/45
Chroma: loaded shard 20/45
Chroma: loaded shard 21/45
Chroma: loaded shard 22/45
Chroma: loaded shard 23/45
Chroma: loaded shard 24/45
Chroma: loaded shard 25/45
Chroma: loaded shard 26/45
Chroma: loaded shard 27/45
Chroma: loaded shard 28/45
Chroma: loaded shard 29/45
Chroma: loaded shard 30/45
Chroma: loaded shard 31/45
Chroma: loaded shard 32/45
Chroma: loaded shard 33/45
Chroma: loaded shard 34/45
Chroma: loaded shard 35/45
Chroma: loaded shard 36/45
Chrom

In [ ]:
import os, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/electronics_kb"
os.makedirs(DRIVE_DIR, exist_ok=True)

# 1. Sanity check before zipping
assert os.path.exists("chroma_db"), "chroma_db not found in the current directory"

# 2. Zip chroma_db
zip_path = "/content/chroma_db.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
    for root, _, files in os.walk("chroma_db"):
        for f in files:
            full = os.path.join(root, f)
            zf.write(full, arcname=full)

print(f"Zip size: {os.path.getsize(zip_path) / 1e9:.2f} GB")

# 3. Copy to Drive
shutil.copy(zip_path, f"{DRIVE_DIR}/chroma_db.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zip size: 30.28 GB


'/content/drive/MyDrive/electronics_kb/chroma_db.zip'

In [ ]:
import os
print(os.getcwd())
print(os.path.exists("chroma_db"))
!ls -la

In [ ]:
# ------------------------------------------------------------
# 4c. Reciprocal Rank Fusion — combine BM25 + dense results
# ------------------------------------------------------------
def reciprocal_rank_fusion(rankings: list, k: int = RRF_K) -> dict:
    """rankings: list of ranked id-lists (one per retrieval method)."""
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

def hybrid_search(query: str, top_k: int = TOP_K_DEFAULT, candidate_k: int = 50) -> list:
    bm25_ids = bm25_search(query, k=candidate_k)
    dense_ids = dense_search(query, k=candidate_k)
    fused = reciprocal_rank_fusion([bm25_ids, dense_ids])
    top_ids = list(fused.keys())[:top_k]

    results = []
    for doc_id in top_ids:
        if doc_id in chunk_id_to_pos:
            row = chunks.iloc[chunk_id_to_pos[doc_id]]
            results.append({
                "chunk_id": doc_id,
                "score": fused[doc_id],
                "parent_asin": row["parent_asin"],
                "chunk_type": row["chunk_type"],
                "aspect": row["aspect"],
                "text": row["text"],
            })
    return results

In [ ]:
# ---------------------------------------------------------------
# Reload helper — so a future session doesn't need to retokenize
# ---------------------------------------------------------------
def load_bm25_index(path: str = BM25_INDEX_PATH):
    global bm25
    with open(path, "rb") as f:
        data = pickle.load(f)
    bm25 = data["bm25"]
    return bm25

if __name__ == "__main__":
    example = hybrid_search("battery life on wireless earbuds", top_k=5)
    for r in example:
        print(r["score"], r["chunk_type"], r["aspect"], r["text"][:100])

In [ ]:
# save result
import json

def save_hybrid_results(queries: list, top_k: int = 10, out_path: str = "hybrid_results.json"):
    all_results = {}
    for q in queries:
        results = hybrid_search(q, top_k=top_k)
        all_results[q] = results  # list of dicts: chunk_id, score, parent_asin, chunk_type, aspect, text
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"Saved {len(queries)} query results -> {out_path}")

# usage
test_queries = [
    "battery life on wireless earbuds",
    "laptop screen resolution",
    "durable phone case",
]
save_hybrid_results(test_queries)

In [ ]:
# save the result
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("hybrid_results.json", "/content/drive/MyDrive/hybrid_results.json")

5. Corpus Analysis
* Corpus statistics table
* Top-K retrieval latency benchmarks
* Recall/Precision baseline
* Token distribution reports

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from config import CHUNKS_PATH, CORPUS_STATS_PATH, TOP_K_DEFAULT

chunks = pd.read_parquet(CHUNKS_PATH)

In [ ]:
# ------------------------------------------------------------
# 5a. Corpus statistics table
# ------------------------------------------------------------
stats = {
    "total_chunks": len(chunks),
    "spec_chunks": int((chunks["chunk_type"] == "spec").sum()),
    "review_chunks": int((chunks["chunk_type"] == "review").sum()),
    "unique_products": chunks["parent_asin"].nunique(),
    "unique_categories": chunks["category"].nunique() if "category" in chunks else None,
    "aspect_distribution": chunks["aspect"].value_counts().to_dict(),
    "avg_chunk_char_len": float(chunks["text"].str.len().mean()),
    "median_chunk_char_len": float(chunks["text"].str.len().median()),
}
print("=== Corpus Statistics ===")
for k, v in stats.items():
    if not isinstance(v, dict):
        print(f"{k}: {v}")
print("Aspect distribution:")
for aspect, count in stats["aspect_distribution"].items():
    print(f"  {aspect}: {count}")


In [ ]:
# ------------------------------------------------------------
# 5b. Token distribution report
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

# sample for speed on large corpora; use full set if corpus is small enough
sample_size = min(20000, len(chunks))
sample = chunks.sample(sample_size, random_state=42)
token_counts = sample["text"].apply(lambda t: len(tokenizer.encode(t, truncation=False)))

stats["token_stats"] = {
    "sample_size": sample_size,
    "mean": float(token_counts.mean()),
    "median": float(token_counts.median()),
    "p90": float(token_counts.quantile(0.9)),
    "p99": float(token_counts.quantile(0.99)),
    "max": int(token_counts.max()),
    "pct_over_512_tokens": float((token_counts > 512).mean() * 100),  # bge context limit
}
print("\n=== Token Distribution ===")
print(stats["token_stats"])

plt.figure(figsize=(8, 4))
plt.hist(token_counts, bins=50)
plt.axvline(512, color="red", linestyle="--", label="embedding model limit (512)")
plt.xlabel("Tokens per chunk")
plt.ylabel("Count")
plt.title("Chunk Token Distribution")
plt.legend()
plt.tight_layout()
plt.savefig("token_distribution.png")
print("Saved token_distribution.png")

In [ ]:
# ------------------------------------------------------------
# 4c. Top-K retrieval latency benchmark
#     Requires step3_index.py's hybrid_search()
# ------------------------------------------------------------
try:
    from step3_index import hybrid_search, bm25_search, dense_search

    test_queries = [
        "battery life on wireless earbuds",
        "laptop screen resolution",
        "durable phone case",
        "noise cancelling headphones quality",
        "cheap usb cable",
    ]

    latency_results = {"bm25": [], "dense": [], "hybrid": []}
    for q in test_queries:
        t0 = time.time(); bm25_search(q, k=TOP_K_DEFAULT); latency_results["bm25"].append(time.time() - t0)
        t0 = time.time(); dense_search(q, k=TOP_K_DEFAULT); latency_results["dense"].append(time.time() - t0)
        t0 = time.time(); hybrid_search(q, top_k=TOP_K_DEFAULT); latency_results["hybrid"].append(time.time() - t0)

    stats["latency_ms"] = {
        method: {"mean": round(np.mean(times) * 1000, 1), "p95": round(np.percentile(times, 95) * 1000, 1)}
        for method, times in latency_results.items()
    }
    print("\n=== Retrieval Latency (ms) ===")
    print(stats["latency_ms"])
except ImportError:
    print("\nSkipping latency benchmark — run step3_index.py in this session first.")

In [ ]:
# ------------------------------------------------------------
# 4d. Recall/Precision baseline
# ------------------------------------------------------------
def precision_at_k(retrieved_ids, relevant_ids, k):
    retrieved_k = retrieved_ids[:k]
    hits = len(set(retrieved_k) & set(relevant_ids))
    return hits / k if k else 0.0

def recall_at_k(retrieved_ids, relevant_ids, k):
    retrieved_k = retrieved_ids[:k]
    hits = len(set(retrieved_k) & set(relevant_ids))
    return hits / len(relevant_ids) if relevant_ids else 0.0

# EXAMPLE
relevance_judgments = {
    # "battery life on wireless earbuds": ["chunk_id_1", "chunk_id_2", ...],
}

if relevance_judgments:
    from step3_index import hybrid_search
    precisions, recalls = [], []
    for query, relevant in relevance_judgments.items():
        retrieved = [r["chunk_id"] for r in hybrid_search(query, top_k=TOP_K_DEFAULT)]
        precisions.append(precision_at_k(retrieved, relevant, TOP_K_DEFAULT))
        recalls.append(recall_at_k(retrieved, relevant, TOP_K_DEFAULT))
    stats["precision_at_k"] = float(np.mean(precisions))
    stats["recall_at_k"] = float(np.mean(recalls))
    print(f"\nPrecision@{TOP_K_DEFAULT}: {stats['precision_at_k']:.3f}")
    print(f"Recall@{TOP_K_DEFAULT}: {stats['recall_at_k']:.3f}")
else:
    print("\nNo relevance_judgments defined yet — recall/precision baseline skipped. "
          "Hand-label ~20-30 (query, relevant_chunk_ids) pairs and fill in "
          "`relevance_judgments` above to get real numbers.")

In [ ]:
# ------------------------------------------------------------
# Save all stats
# ------------------------------------------------------------
with open(CORPUS_STATS_PATH, "w") as f:
    json.dump(stats, f, indent=2, default=str)
print(f"\nSaved all stats -> {CORPUS_STATS_PATH}")